In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from processing import *
from wavelengths import *
from reprojection import *
from datetime import datetime

In [2]:
q_V = 299792458 / 6173.341
q_B = q_V * 0.231
tuning_constant = 3.513e-4
temperature_constant = 4.01225e-2
alpha = 1.327124e20

In [3]:
files = sorted(glob.glob('/home/ulyanov/data/solo/phi/2026/blos/*.fits'))

In [4]:
files[1980]

'/home/ulyanov/data/solo/phi/2026/blos/phi-fdt-blos_20260527T050009_V202609091813C_0645270502.fits'

In [53]:
with fits.open(files[1978]) as hdul:
    data1 = hdul[0].data
    header1 = hdul[0].header
    fg_data = hdul['PHI_FITS_FG_settings'].data
    pmp_data = hdul['PHI_FITS_PMP_settings'].data


velocity = header1['OBS_VR']
temperature = header1['FGOV1PT1']
contposn = header1['CONTPOSN']

print(velocity, temperature, contposn)
print(np.nanmedian(data1) / q_V)

-15.1958409995032 60.99 blue
-6.4462815e-06


In [54]:
with fits.open(files[1979]) as hdul:
    data2 = hdul[0].data
    header2 = hdul[0].header
    fg_data = hdul['PHI_FITS_FG_settings'].data
    pmp_data = hdul['PHI_FITS_PMP_settings'].data


velocity = header2['OBS_VR']
temperature = header2['FGOV1PT1']
contposn = header2['CONTPOSN']

print(velocity, temperature, contposn)
print(np.nanmedian(data2) / q_V)

-55.3331284914268 60.99 red
-3.73503e-06


In [55]:
data2 = reproject(data2, header2, header1, correct_mu=True)

data1 = rebin(data1, 4)
data2 = rebin(data2, 4)

In [56]:
plt.figure(figsize=(10,10))
plt.imshow(data2 - data1, 'seismic', vmin=-100, vmax=100)
plt.tight_layout()

In [57]:
t = ~np.isnan(data1) & ~np.isnan(data2) & (np.abs(data1) < 200) & (np.abs(data2) < 200)

plt.figure(figsize=(10,10))
plt.plot(data1[t], data2[t], '.', ms=0.5)
plt.plot([-500,500], [-500,500], 'gray', lw=0.5)

plt.xlim(-500,500)
plt.ylim(-500,500)
plt.tight_layout()

In [58]:
t = ~np.isnan(data1) & ~np.isnan(data2)

x = data1[t].copy()
y = data2[t].copy()

A = np.array([[np.mean(x ** 2), np.mean(x * y)],
              [np.mean(x * y), np.mean(y ** 2)]])

vals, vecs = np.linalg.eigh(A)
u, v = vecs[:, 1]

print(v / u)

0.97204906
